#Training

**Pipeline:**
1. Load dataset & create DataLoader
2. Instantiate Generator, Discriminator, VGG feature extractor from `model.py`
3. Train with adversarial + perceptual loss
4. Track & plot loss curves
5. Save checkpoints
6. Quick inference sanity check

In [ ]:
!pip install torch torchvision pillow tqdm matplotlib -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports & Configuration

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

# Import model components from model.py
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from model import CTScanDataset, Generator, Discriminator, VGGFeatureExtractor


#  CONFIGURATION

DATASET_PATH   = r"C:\Users\NarisettyDharmaTeja\Downloads\Dataset"
CHECKPOINT_DIR = "checkpoints"
BATCH_SIZE     = 8
EPOCHS         = 10
LEARNING_RATE  = 1e-4
ADV_WEIGHT     = 1e-3    # Weight for adversarial loss in G total loss

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Dataset

In [ ]:
dataset = CTScanDataset(DATASET_PATH)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f"Total images : {len(dataset)}")
print(f"Batches/epoch: {len(loader)}")
print(f"Batch size   : {BATCH_SIZE}")

## 3. Instantiate Models
Import the **Generator**, **Discriminator**, and **VGG Feature Extractor** from `model.py`.

In [ ]:
# Instantiate models
generator     = Generator().to(device)
discriminator = Discriminator().to(device)
vgg_extractor = VGGFeatureExtractor().to(device)

# Loss functions
bce_loss = nn.BCELoss()
mse_loss = nn.MSELoss()

# Optimisers
opt_g = optim.Adam(generator.parameters(),     lr=LEARNING_RATE)
opt_d = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE)

# Print model sizes
def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Generator parameters     : {count_params(generator):,}")
print(f"Discriminator parameters : {count_params(discriminator):,}")
print(f"VGG extractor (frozen)   : {sum(p.numel() for p in vgg_extractor.parameters()):,}")

## 4. Model Architecture Summary
Verify the Generator and Discriminator architectures before training.

In [ ]:
# Quick forward-pass sanity check
with torch.no_grad():
    dummy_lr = torch.randn(1, 3, 64, 64).to(device)
    dummy_sr = generator(dummy_lr)
    dummy_d  = discriminator(dummy_sr)

print(f"Generator:     {dummy_lr.shape}  →  {dummy_sr.shape}")
print(f"Discriminator: {dummy_sr.shape}  →  {dummy_d.shape}")
print(f"\nGenerator Architecture:\n{'='*50}")
print(generator)
print(f"\nDiscriminator Architecture:\n{'='*50}")
print(discriminator)

## 5. Training Loop
**Generator loss** = perceptual (VGG feature MSE) + 1e-3 × adversarial (BCE)

**Discriminator loss** = 0.5 × (real_BCE + fake_BCE)

Loss history is logged per epoch for plotting.

In [ ]:

#  Training history

history = {
    "g_loss": [],  "d_loss": [],
    "g_adv":  [],  "g_perc": [],
    "d_real": [],  "d_fake": [],
}

for epoch in range(EPOCHS):
    g_losses, d_losses = [], []
    g_advs, g_percs    = [], []
    d_reals, d_fakes   = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for lr, hr in pbar:
        lr = lr.to(device)
        hr = hr.to(device)
        bs = hr.size(0)

        # Generate super-resolved image
        fake = generator(lr)

        real_labels = torch.ones(bs, device=device)
        fake_labels = torch.zeros(bs, device=device)

        # Train Discriminator 
        real_pred = discriminator(hr)
        fake_pred = discriminator(fake.detach())

        d_real_loss = bce_loss(real_pred, real_labels)
        d_fake_loss = bce_loss(fake_pred, fake_labels)
        d_loss = (d_real_loss + d_fake_loss) / 2

        opt_d.zero_grad()
        d_loss.backward()
        opt_d.step()

        # Train Generator 
        adv_loss  = bce_loss(discriminator(fake), real_labels)
        perc_loss = mse_loss(vgg_extractor(fake), vgg_extractor(hr))
        g_loss    = perc_loss + ADV_WEIGHT * adv_loss

        opt_g.zero_grad()
        g_loss.backward()
        opt_g.step()

        # Log batch losses
        g_losses.append(g_loss.item())
        d_losses.append(d_loss.item())
        g_advs.append(adv_loss.item())
        g_percs.append(perc_loss.item())
        d_reals.append(d_real_loss.item())
        d_fakes.append(d_fake_loss.item())

        pbar.set_postfix(G=f"{g_loss.item():.4f}", D=f"{d_loss.item():.4f}")

    # Epoch averages
    history["g_loss"].append(np.mean(g_losses))
    history["d_loss"].append(np.mean(d_losses))
    history["g_adv"].append(np.mean(g_advs))
    history["g_perc"].append(np.mean(g_percs))
    history["d_real"].append(np.mean(d_reals))
    history["d_fake"].append(np.mean(d_fakes))

    print(f"Epoch {epoch+1:>2}/{EPOCHS}  │  "
          f"G Loss: {history['g_loss'][-1]:.4f} (perc: {history['g_perc'][-1]:.4f}, adv: {history['g_adv'][-1]:.4f})  │  "
          f"D Loss: {history['d_loss'][-1]:.4f} (real: {history['d_real'][-1]:.4f}, fake: {history['d_fake'][-1]:.4f})")

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            "epoch": epoch + 1,
            "generator": generator.state_dict(),
            "discriminator": discriminator.state_dict(),
            "opt_g": opt_g.state_dict(),
            "opt_d": opt_d.state_dict(),
            "history": history,
        }, os.path.join(CHECKPOINT_DIR, f"srgan_epoch_{epoch+1}.pth"))
        print(f"Checkpoint saved: srgan_epoch_{epoch+1}.pth")

## 6. Save Final Model

In [ ]:
# Save final model weights and full training history
final_path = os.path.join(CHECKPOINT_DIR, "srgan_final.pth")

torch.save({
    "epoch": EPOCHS,
    "generator": generator.state_dict(),
    "discriminator": discriminator.state_dict(),
    "opt_g": opt_g.state_dict(),
    "opt_d": opt_d.state_dict(),
    "history": history,
}, final_path)

print(f"Final model saved to: {final_path}")
print(f"File size: {os.path.getsize(final_path) / (1024**2):.1f} MB")

## 7. Training Loss Curves
Visualize generator and discriminator loss progression over epochs.

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Overall G vs D loss
axes[0].plot(epochs_range, history["g_loss"], "o-", label="Generator", color="royalblue")
axes[0].plot(epochs_range, history["d_loss"], "s-", label="Discriminator", color="crimson")
axes[0].set_title("Generator vs Discriminator Loss", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (b) Generator loss breakdown
axes[1].plot(epochs_range, history["g_perc"], "o-", label="Perceptual (VGG)", color="teal")
axes[1].plot(epochs_range, history["g_adv"],  "s-", label="Adversarial (BCE)", color="orange")
axes[1].set_title("Generator Loss Breakdown", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# (c) Discriminator real vs fake
axes[2].plot(epochs_range, history["d_real"], "o-", label="Real loss", color="green")
axes[2].plot(epochs_range, history["d_fake"], "s-", label="Fake loss", color="red")
axes[2].axhline(0.693, linestyle="--", color="gray", alpha=0.5, label="Random guess (ln2)")
axes[2].set_title("Discriminator: Real vs Fake", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("BCE Loss")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Quick Inference Sanity Check
Run the trained generator on a few samples to verify output quality before full evaluation.

In [ ]:
generator.eval()

lr_batch, hr_batch = next(iter(loader))
lr_batch = lr_batch.to(device)

with torch.no_grad():
    sr_batch = generator(lr_batch)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
titles = ["LR Input (64*64)", "SR Output (256*256)", "HR Ground Truth (256*256)"]

for i in range(4):
    lr_img = lr_batch[i].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    sr_img = sr_batch[i].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    hr_img = hr_batch[i].permute(1, 2, 0).clamp(0, 1).numpy()

    axes[0, i].imshow(lr_img)
    axes[1, i].imshow(sr_img)
    axes[2, i].imshow(hr_img)

    for row in range(3):
        axes[row, i].axis("off")
        if i == 0:
            axes[row, i].set_ylabel(titles[row], fontsize=11, fontweight="bold")

fig.suptitle("Training Sanity Check:  LR -> SR -> HR", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()
